# Simple Agentic AI with Google Gemini + FastAPI

This Google Colab tutorial builds a small **course-support agent** and converts it into a FastAPI service.

The agent can:

- answer general questions using Google Gemini;
- search a small course catalogue through a tool;
- calculate course discounts through a tool;
- check a sample order through a tool;
- decide by itself which tool to call;
- expose the agent through `POST /chat`;
- validate and test the API without consuming Gemini quota.

## Architecture

```text
User / Client
     |
     v
FastAPI POST /chat
     |
     v
Gemini Agent ------> Course Search Tool
     |--------------> Discount Tool
     |--------------> Order Status Tool
     |
     v
JSON Response
```

> The tools use sample in-memory data for learning. Replace them with database, SAP, CRM, or other API calls in a real solution.

## 1. Install the required packages

- `google-genai`: official Google Gen AI Python SDK
- `fastapi`: API framework
- `uvicorn`: ASGI server
- `httpx`: required by FastAPI's test client
- `pyngrok`: optional public URL for Colab

In [ ]:
%pip install -q -U google-genai fastapi uvicorn httpx pyngrok

## 2. Load the Gemini API key securely

1. Create a key in [Google AI Studio](https://aistudio.google.com/apikey).
2. In Colab, open **Secrets** from the left sidebar.
3. Add a secret named `GEMINI_API_KEY`.
4. Allow notebook access to the secret.

The key is never printed.

In [ ]:
import os

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except (ImportError, KeyError):
    from getpass import getpass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or getpass("Enter GEMINI_API_KEY: ")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not provided.")

print("Gemini API key loaded securely.")

## 3. Create sample business data

These dictionaries act as a temporary database. They keep the tutorial self-contained.

In [ ]:
COURSES = [
    {
        "course_id": "C101",
        "name": "Python for Generative AI",
        "category": "Python",
        "level": "Beginner",
        "price_inr": 5000,
    },
    {
        "course_id": "C102",
        "name": "Agentic AI with LangGraph",
        "category": "Agentic AI",
        "level": "Intermediate",
        "price_inr": 8000,
    },
    {
        "course_id": "C103",
        "name": "SAP Business AI Fundamentals",
        "category": "SAP Business AI",
        "level": "Beginner",
        "price_inr": 10000,
    },
    {
        "course_id": "C104",
        "name": "FastAPI for AI Applications",
        "category": "API Development",
        "level": "Intermediate",
        "price_inr": 6000,
    },
]

ORDERS = {
    "ORD-1001": {"status": "confirmed", "course_id": "C103", "learner": "Amit"},
    "ORD-1002": {"status": "payment_pending", "course_id": "C102", "learner": "Neha"},
    "ORD-1003": {"status": "completed", "course_id": "C101", "learner": "Rahul"},
}

print(f"Loaded {len(COURSES)} courses and {len(ORDERS)} sample orders.")

## 4. Define the agent tools

A tool is an ordinary Python function that the model is allowed to request. Clear type hints and docstrings help Gemini understand when and how to use it.

In [ ]:
def search_courses(query: str) -> dict:
    """Searches available courses by name, category, or level.

    Args:
        query: Search text such as SAP, beginner, Python, FastAPI, or agentic AI.

    Returns:
        A dictionary containing the number of matches and matching courses.
    """
    query_lower = query.strip().lower()
    matches = [
        course
        for course in COURSES
        if query_lower in course["name"].lower()
        or query_lower in course["category"].lower()
        or query_lower in course["level"].lower()
    ]
    return {"count": len(matches), "courses": matches}


def calculate_discount(course_id: str, discount_percent: float) -> dict:
    """Calculates the final course price after a percentage discount.

    Args:
        course_id: Course identifier such as C101 or C103.
        discount_percent: Discount percentage from 0 through 50.

    Returns:
        Original price, discount amount, and final price in Indian rupees.
    """
    course = next((item for item in COURSES if item["course_id"].upper() == course_id.upper()), None)
    if course is None:
        return {"error": f"Course {course_id} was not found."}
    if not 0 <= discount_percent <= 50:
        return {"error": "Discount must be between 0 and 50 percent."}

    original_price = course["price_inr"]
    discount_amount = round(original_price * discount_percent / 100, 2)
    return {
        "course_id": course["course_id"],
        "course_name": course["name"],
        "original_price_inr": original_price,
        "discount_percent": discount_percent,
        "discount_amount_inr": discount_amount,
        "final_price_inr": round(original_price - discount_amount, 2),
    }


def get_order_status(order_id: str) -> dict:
    """Returns the status of a course order.

    Args:
        order_id: Order identifier such as ORD-1001.

    Returns:
        Order details or an error when the order is unavailable.
    """
    order = ORDERS.get(order_id.upper())
    if order is None:
        return {"error": f"Order {order_id} was not found."}
    return {"order_id": order_id.upper(), **order}


AGENT_TOOLS = [search_courses, calculate_discount, get_order_status]

print(search_courses("SAP"))
print(calculate_discount("C103", 10))
print(get_order_status("ORD-1002"))

## 5. Create the Gemini agent

The Google Gen AI SDK can automatically:

1. convert Python functions into tool schemas;
2. let Gemini choose an appropriate tool;
3. execute the selected Python function;
4. send the tool result back to Gemini;
5. return a natural-language final answer.

That tool-selection and execution loop is what makes this a simple agent rather than a plain chatbot.

In [ ]:
from google import genai
from google.genai import types

MODEL_NAME = "gemini-2.5-flash"
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

SYSTEM_INSTRUCTION = """
You are a concise course-support agent for an AI training company.
Use the available tools whenever the user asks about courses, prices,
discount calculations, or order status. Never invent a course, price,
order, or tool result. If a tool returns an error, explain it clearly.
All prices are in Indian rupees. Keep the final response helpful and concise.
""".strip()


def run_agent(message: str) -> str:
    """Sends one user message to Gemini with access to local Python tools."""
    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=message,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=AGENT_TOOLS,
            temperature=0.2,
        ),
    )
    if not response.text:
        raise RuntimeError("Gemini returned an empty response.")
    return response.text

print("Agent created with model:", MODEL_NAME)

## 6. Test the agent directly

Each question encourages a different behaviour: tool selection, tool execution, or a direct response.

In [ ]:
sample_questions = [
    "Which beginner SAP course is available?",
    "What will course C103 cost after a 15 percent discount?",
    "What is the status of order ORD-1002?",
]

for question in sample_questions:
    print("\nUSER:", question)
    print("AGENT:", run_agent(question))

## 7. Convert the agent into a FastAPI application

The request and response models provide validation and automatic API documentation. The agent is supplied through a FastAPI dependency, which also makes unit testing easy.

In [ ]:
from typing import Callable

from fastapi import Depends, FastAPI, HTTPException
from pydantic import BaseModel, Field


class ChatRequest(BaseModel):
    message: str = Field(
        min_length=2,
        max_length=1000,
        examples=["What is the price of the SAP Business AI course after a 10% discount?"],
    )


class ChatResponse(BaseModel):
    answer: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


def get_agent_service() -> Callable[[str], str]:
    """FastAPI dependency that provides the Gemini agent function."""
    return run_agent


app = FastAPI(
    title="Gemini Course Support Agent API",
    description="A simple tool-using agent built with Google Gemini and FastAPI.",
    version="1.0.0",
)


@app.get("/", tags=["General"])
def home():
    return {
        "message": "Gemini Course Support Agent API",
        "documentation": "/docs",
    }


@app.get("/health", response_model=HealthResponse, tags=["General"])
def health():
    return {"status": "healthy", "model": MODEL_NAME}


@app.post("/chat", response_model=ChatResponse, tags=["Agent"])
def chat(
    request: ChatRequest,
    agent_service: Callable[[str], str] = Depends(get_agent_service),
):
    try:
        answer = agent_service(request.message)
        return ChatResponse(answer=answer, model=MODEL_NAME)
    except Exception as exc:
        # Log the full exception internally in a production application.
        raise HTTPException(
            status_code=503,
            detail="The AI service is temporarily unavailable.",
        ) from exc


print("FastAPI application created.")

## 8. Test the API without calling Gemini

The test replaces the real Gemini dependency with a deterministic fake function. This tests routing, validation, response structure, and error handling without API cost.

In [ ]:
from fastapi.testclient import TestClient


def fake_agent(message: str) -> str:
    return f"Mock agent received: {message}"


app.dependency_overrides[get_agent_service] = lambda: fake_agent
test_client = TestClient(app)

health_response = test_client.get("/health")
assert health_response.status_code == 200
assert health_response.json()["status"] == "healthy"

chat_response = test_client.post(
    "/chat",
    json={"message": "Show me an SAP AI course"},
)
assert chat_response.status_code == 200
assert "Mock agent received" in chat_response.json()["answer"]

invalid_response = test_client.post("/chat", json={"message": ""})
assert invalid_response.status_code == 422

print("Health response:", health_response.json())
print("Chat response:", chat_response.json())
print("Invalid request status:", invalid_response.status_code)
print("✅ All FastAPI tests passed.")

# Restore the real Gemini agent for subsequent cells.
app.dependency_overrides.clear()

## 9. Call the real `/chat` API in memory

This request goes through FastAPI and then calls the real Gemini agent. It consumes Gemini quota.

In [ ]:
live_client = TestClient(app)

response = live_client.post(
    "/chat",
    json={"message": "Calculate a 20 percent discount for course C104."},
)

print("Status code:", response.status_code)
print("Response:", response.json())

## 10. Optional: publish the API from Colab

Colab cannot directly expose its local port. The following cell starts Uvicorn in a background thread and creates a temporary ngrok URL.

1. Create a free ngrok account.
2. Copy your auth token.
3. Run the cell and enter the token securely.
4. Open the displayed `/docs` URL.

> Anyone with the temporary URL can call this demonstration endpoint and consume your Gemini quota. Stop the tunnel when finished. Add authentication before using it beyond a controlled demo.

In [ ]:
# Uncomment and run this cell only when you need a public demonstration URL.

# import threading
# import time
# from getpass import getpass
#
# import uvicorn
# from pyngrok import ngrok
#
# NGROK_AUTH_TOKEN = getpass("Enter your ngrok auth token: ")
# ngrok.set_auth_token(NGROK_AUTH_TOKEN)
#
# server = uvicorn.Server(
#     uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
# )
# server_thread = threading.Thread(target=server.run, daemon=True)
# server_thread.start()
# time.sleep(2)
#
# public_url = ngrok.connect(8000).public_url
# print("API base URL:", public_url)
# print("Swagger UI:", public_url + "/docs")
# print("Health endpoint:", public_url + "/health")

## 11. Example API request

Replace `YOUR_PUBLIC_URL` with the ngrok URL printed above.

```bash
curl -X POST "YOUR_PUBLIC_URL/chat" \
  -H "Content-Type: application/json" \
  -d '{"message":"Which beginner AI courses are available?"}'
```

Example response structure:

```json
{
  "answer": "The available beginner course is ...",
  "model": "gemini-2.5-flash"
}
```

## 12. How the agentic flow works

For the request **“Calculate a 15% discount for course C103”**:

1. FastAPI validates the JSON request.
2. The `/chat` endpoint sends the message to `run_agent()`.
3. Gemini sees the available tool descriptions.
4. Gemini selects `calculate_discount` and supplies `course_id="C103"` and `discount_percent=15`.
5. The SDK executes the Python function.
6. The function returns structured price data.
7. Gemini converts the tool result into a natural-language answer.
8. FastAPI returns the answer as validated JSON.

## Production improvements

- store conversations in a database or session store;
- use async endpoints and the asynchronous Gen AI client;
- replace sample dictionaries with SAP HANA, PostgreSQL, or enterprise APIs;
- add JWT/OAuth authentication and role-based authorization;
- add rate limiting, request IDs, structured logs, and monitoring;
- do not expose internal exception details to API consumers;
- add timeouts, retries, safety controls, and human approval for high-impact actions;
- containerise with Docker and deploy to SAP BTP, Google Cloud Run, or another platform.

## Useful official references

- [Google Gen AI SDK for Python](https://googleapis.github.io/python-genai/)
- [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)
- [FastAPI documentation](https://fastapi.tiangolo.com/)